# Decision Tree Optimization

In [43]:
import pandas as pd
import json

In [44]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [45]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Mid point

In [46]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    print(mid_point)
    mid_points.append(mid_point)

1.5
2.5
3.5
4.5
5.5
6.5
7.5
8.5
9.5
10.5
11.5


In [47]:
x = df['studied'].values
x

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [48]:
print("from 1st item to 2nd last:")
print(x[:-1])
print("from 2nd item to last:")
print(x[1:])

from 1st item to 2nd last:
[ 1  2  3  4  5  6  7  8  9 10 11]
from 2nd item to last:
[ 2  3  4  5  6  7  8  9 10 11 12]


In [49]:
mid_points = (x[:-1] + x[1:]) / 2
print(mid_points)

[ 1.5  2.5  3.5  4.5  5.5  6.5  7.5  8.5  9.5 10.5 11.5]


## Sliding Windows

### Gini Formula

In [50]:
def gini(y):
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)
    gini = 1 - (p0**2 + p1**2)
    return gini

In [51]:
# test gini
gini(df['passed'].values)

np.float64(0.4444444444444444)

In [52]:
def weighted_gini(y_left, y_right):
    n = len(y_left) + len(y_right)
    gini_left = gini(y_left)
    gini_right = gini(y_right)
    weighted_gini = (len(y_left) / n) * gini_left + (len(y_right) / n) * gini_right
    return weighted_gini

In [56]:
# Split base on first mid point
mid_point = mid_points[0]
left_node = df[df['studied'] <= mid_point]
right_node = df[df['studied'] > mid_point]
weighted_gini_first = weighted_gini(left_node['passed'].values, right_node['passed'].values)

In [57]:
print(mid_point)
print(left_node)
print(right_node)

1.5
   studied  passed
0        1       0
    studied  passed
1         2       0
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1


In [59]:
# second mid point split
mid_point = mid_points[1]
left_node = df[df['studied'] <= mid_point]
right_node = df[df['studied'] > mid_point]
print(mid_point)
print(left_node)
print(right_node)
weighted_gini_second = weighted_gini(left_node['passed'].values, right_node['passed'].values)


2.5
   studied  passed
0        1       0
1        2       0
    studied  passed
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1


In [60]:
print("Weighted Gini for first split:", weighted_gini_first)
print("Weighted Gini for second split:", weighted_gini_second)

Weighted Gini for first split: 0.36363636363636365
Weighted Gini for second split: 0.26666666666666655


## Sliding Windows

Pseudo code:

1. Sort feature
2. Initialize LEFT counts to zero
3. Initialize RIGHT counts to total class counts
4. Move one sample at a time from RIGHT to LEFT
5. Update counts
6. Compute weighted Gini
7. Track best boundary
8. Convert best boundary to midpoint threshold




In [68]:
import numpy as np

In [96]:
def best_gini_search(df):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # initialized count
    left_p0_count = 0
    left_p1_count = 0
    right_p0_count = (y==0).sum()
    right_p1_count = (y==1).sum()

    # print(left_p0_count)
    # print(left_p1_count)
    # print(right_p0_count)
    # print(right_p1_count)

    # initializa total number of data
    n = len(y)
    n_right = n
    n_left = 0

    for i in np.arange(n-1):
        if y[i] == 0:
            left_p0_count += 1
            right_p0_count -= 1
            n_left += 1
            n_right -= 1
        else:
            left_p1_count += 1
            right_p1_count -= 1
            n_left += 1
            n_right -= 1

        print(left_p0_count)
        print(left_p1_count)
        print(n_left)
        
        weighted_gini = (n_left / n) *  (1 - (left_p0_count/n_left**2 + left_p1_count/n_left**2)) + (n_right / n) * (1 - (right_p0_count/n_right**2 + right_p1_count/n_right**2))
        print(weighted_gini)
 
        if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_split_value = (x[i] + x[i+1])/2
            #print(best_split_value)
            print(best_gini)
        
    return best_split_value, best_gini


In [97]:
best_gini_search(df)

1
0
1
0.8333333333333333
0.8333333333333333
2
0
2
0.8333333333333334
3
0
3
0.8333333333333333
4
0
4
0.8333333333333333
4
1
5
0.8333333333333335
4
2
6
0.8333333333333334
4
3
7
0.8333333333333335
4
4
8
0.8333333333333333
4
5
9
0.8333333333333333
4
6
10
0.8333333333333334
4
7
11
0.8333333333333333


(np.float64(1.5), np.float64(0.8333333333333333))